# Streamlit + optional ngrok demo

This Colab example starts a minimal Streamlit presentation and opens an explicit TLS tunnel. A tunnel publicly exposes the service: use only non-sensitive demo data, never hardcode a token, and stop it after use. Kaggle may prohibit outbound tunnels.

In [ ]:
%pip install -q "streamlit==1.60.0" "pyngrok==8.1.2"

In [ ]:
%%writefile course_streamlit.py
import streamlit as st
st.set_page_config(page_title="EnterpriseRAG Course Demo")
st.title("EnterpriseRAG Course Demo")
st.warning("Demo data only. This temporary surface is publicly tunneled.")
temperature = st.slider("Temperature", 0.0, 2.0, 0.1)
top_k = st.slider("Generation top_k", 0, 200, 50)
top_p = st.slider("top_p", 0.05, 1.0, 0.9)
st.json({"temperature": temperature, "generation_top_k": top_k, "top_p": top_p})

In [ ]:
import os
import subprocess

from pyngrok import conf, ngrok

token = os.getenv("NGROK_AUTHTOKEN")
if not token:
    raise RuntimeError(
        "Add NGROK_AUTHTOKEN through Colab Secrets or the environment; never paste it into this notebook."
    )
conf.get_default().auth_token = token
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "course_streamlit.py",
        "--server.headless=true",
        "--server.port=8501",
    ]
)
try:
    tunnel = ngrok.connect(8501, bind_tls=True)
    print("Temporary demo URL:", tunnel.public_url)
except Exception:
    process.terminate()
    raise

In [ ]:
# Run this cleanup cell when the presentation ends.
try:
    ngrok.disconnect(tunnel.public_url)
    ngrok.kill()
finally:
    process.terminate()
print("Tunnel and Streamlit process stopped.")

## Expected output

The launch cell prints one temporary HTTPS URL. The cleanup cell confirms shutdown. If tunneling is restricted, the launch cell raises an error and terminates Streamlit instead of attempting a bypass.